# Model Exploration

This notebook demonstrates loading, navigating, analyzing, and visualizing a SysML v2 model using **sysmlpy**.

Make sure you have installed dependencies with `hatch install` first.

In [ ]:
from pathlib import Path
from sysmlpy import load_project, analyze
from sysmlpy.plantuml import (
    as_general_view,
    as_interconnection_view,
    as_action_flow_view,
    as_package_view,
    as_tree_diagram,
)

## 1. Load the Model

In [ ]:
models_dir = Path('../sysml/models')
model = load_project(str(models_dir))
print(f'Loaded model with {len(model)} top-level element(s)')

## 2. Interactive Tree View

sysmlpy models render as collapsible HTML trees in Jupyter:

In [ ]:
model

## 3. Semantic Analysis

Run the semantic analyzer to check for undefined symbols, import resolution issues, and naming conventions.

In [ ]:
result = analyze(model, style_checks=True)

if result.errors:
    print(f'{len(result.errors)} error(s):')
    for issue in result.errors:
        print(f'  [{issue.code}] {issue.message}')
else:
    print('No errors found.')

if result.warnings:
    print(f'{len(result.warnings)} warning(s):')
    for issue in result.warnings:
        print(f'  [{issue.code}] {issue.message}')
else:
    print('No warnings found.')

## 4. Navigate the Model

Use typed accessors and `find()` / `find_one()` to query elements.

In [ ]:
# List all packages
for pkg in model.packages:
    print(f'Package: {pkg.name}')
    for child in pkg:
        print(f'  {child.sysml_type}: {child.name}')

In [ ]:
# Find all part definitions
parts = model.find(sysml_type='part def')
print(f'Found {len(parts)} part definition(s):')
for part in parts:
    print(f'  {part.name}')

In [ ]:
# Find a specific element by name
camera = model.find_one('Camera')
if camera:
    print(f'Camera has {len(camera)} child element(s):')
    for child in camera:
        print(f'  {child.sysml_type}: {child.name}')

## 5. Generate PlantUML Diagrams

sysmlpy can generate 17 different PlantUML view types. Here are a few common ones.

In [ ]:
# General View
print(as_general_view(model, style='bw'))

In [ ]:
# Action Flow View
print(as_action_flow_view(model, style='bw'))

In [ ]:
# Package View
print(as_package_view(model, style='bw'))

In [ ]:
# Tree Diagram
print(as_tree_diagram(model, style='bw'))

## 6. Save Diagrams to Files

You can also write the PlantUML output to `.puml` files and render them with the PlantUML CLI or VS Code extension.

In [ ]:
output_dir = Path('../output')
output_dir.mkdir(exist_ok=True)

views = {
    'general_view': as_general_view,
    'interconnection_view': as_interconnection_view,
    'action_flow_view': as_action_flow_view,
    'package_view': as_package_view,
    'tree_diagram': as_tree_diagram,
}

for name, func in views.items():
    puml = func(model, style='bw')
    (output_dir / f'{name}.puml').write_text(puml, encoding='utf-8')
    print(f'Wrote {name}.puml')

print(f'\nRender with: plantuml {output_dir}/*.puml')